In [ ]:
if y_injected is not None:
    X_eval = X_scored.copy()
    X_eval["is_injected_anomaly"] = y_injected.values

    top_n = 25
    top_hits = X_eval.sort_values("anomaly_score").head(top_n)
    hit_rate = (top_hits["is_injected_anomaly"].sum() / top_n) * 100

    print(f"Top {top_n} anomaly hit-rate (injected anomalies): {hit_rate:.1f}%")
    print(top_hits[["timestamp_utc","username","source_ip","country","result","anomaly_score","is_injected_anomaly"]].head(25))
else:
    print("No injected anomaly labels found (this is fine).")


In [ ]:
cols = [
    "timestamp_utc","username","event_source","auth_type",
    "source_ip","ip_prefix","country","result","failure_reason",
    "hour","is_offhours","ip_is_private","ip_is_reserved",
    "anomaly_score",
]

top = X_scored.sort_values("anomaly_score").head(25)[cols]
top


In [ ]:
pre = build_preprocessor(cfg=cfg)

model = IsolationForest(
    n_estimators=250,
    contamination=0.02,   # ~2% anomalies expected; tune later
    random_state=42,
)

pipe = Pipeline(steps=[
    ("preprocess", pre),
    ("model", model),
])

pipe.fit(X_df)

# IsolationForest: higher score_samples == more normal; lower == more anomalous
scores = pipe.named_steps["model"].score_samples(pipe.named_steps["preprocess"].transform(X_df))
X_scored = X_df.copy()
X_scored["anomaly_score"] = scores

X_scored.sort_values("anomaly_score").head(10)


In [ ]:
cfg = FeatureConfig(one_hot_min_frequency=5)

X_df, y_injected = make_xy(df_raw, cfg=cfg)

# sanity check
X_df[["timestamp_utc","username","source_ip","country","result","hour","is_offhours","ip_is_private","ip_prefix"]].head()


In [ ]:
import pandas as pd
from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.ensemble import IsolationForest

from src.preprocess import load_logs, make_xy, build_preprocessor, FeatureConfig

DATA_PATH = Path("poc-auth-anomaly/data/sample_auth_logs.csv")

df_raw = load_logs(DATA_PATH)
df_raw.head()
